# 09b Raw View Window Validation

Official audit notebook for validating whether the 광일 master usage/content features are based on raw view day0~20.

In [1]:

from pathlib import Path
from datetime import datetime
import subprocess, os, re, zipfile
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)

STEP='09b_raw_view_window_validation_260514'
EXPECTED={'C:/Code/ott-churn-prediction', r'C:\Code\ott-churn-prediction'}
actual_root=subprocess.check_output(['git','rev-parse','--show-toplevel'], text=True).strip()
if actual_root not in EXPECTED:
    raise SystemExit(f'REPO ROOT MISMATCH: {actual_root}. No validation outputs written.')
ROOT=Path(actual_root.replace('/', os.sep)).resolve(); PARK=(ROOT/'park.ingyeom').resolve(); DATA=PARK/'data'
NOTE=PARK/'note.md'; NB_PATH=PARK/'notebook'/STEP/f'{STEP}.ipynb'
BASE_OUT=PARK/'reports'/'audits'/STEP
OUT=BASE_OUT/('run_'+datetime.now().strftime('%Y%m%d_%H%M%S')) if BASE_OUT.exists() and any(BASE_OUT.iterdir()) else BASE_OUT
ZIP_DIR=PARK/'zip'; ZIP_PATH=ZIP_DIR/f'{STEP}_review_package.zip'
IN={'master':DATA/'(광일)Membership_v2_with_derived_features.csv','membership':DATA/'Membership_train.csv','view':DATA/'View_History_v2.csv','mapping':DATA/'User_Mapping_v2.csv','movie':DATA/'Movie_Master_v2.csv','prev06':PARK/'reports'/'audits'/'06_common_preprocessing_and_final_cohort_260513'/'06_final_checks.csv','prev09':PARK/'reports'/'eda'/'09_promotion_repurchase_2x2_eda_260513'/'09_final_checks.csv','note':NOTE}
SRC=[IN[k] for k in ['master','membership','view','mapping','movie']]
mt_before={str(p):p.stat().st_mtime_ns if p.exists() else None for p in SRC}

def inside(x,parent=PARK):
    x=Path(x).resolve(); parent=Path(parent).resolve(); return x==parent or parent in x.parents

def wcsv(df,name):
    OUT.mkdir(parents=True, exist_ok=True); p=OUT/name; df.to_csv(p,index=False,encoding='utf-8-sig'); return p

def dt(s):
    if pd.api.types.is_numeric_dtype(s): return pd.to_datetime(s.astype('Int64').astype(str), format='%Y%m%d', errors='coerce')
    return pd.to_datetime(s, errors='coerce')

def target(s):
    return s.map(lambda x: 1 if str(x).strip().upper() in {'1','Y','YES','TRUE','T'} else (0 if str(x).strip().upper() in {'0','N','NO','FALSE','F'} else np.nan))

def det(cols,cands):
    cols=list(cols); low={str(c).lower():c for c in cols}
    for c in cands:
        if c in cols: return c
        if c.lower() in low: return low[c.lower()]
    for col in cols:
        if any(c.lower() in str(col).lower() for c in cands): return col
    return None

def cmpnum(mdf,rdf,mcol,rcol,tol=1e-6):
    comp=mdf[['source_row_number','USER_KEY','reg_date','end_date',mcol]].merge(rdf[['source_row_number',rcol]],on='source_row_number',how='left')
    comp[rcol]=comp[rcol].fillna(0); a=pd.to_numeric(comp[mcol],errors='coerce').fillna(0); b=pd.to_numeric(comp[rcol],errors='coerce').fillna(0); d=a-b; ok=d.abs()<=tol
    return comp, {'master_column':mcol,'raw_recomputed_column':rcol,'compared_row_count':len(comp),'exact_match_count':int(ok.sum()),'mismatch_count':int((~ok).sum()),'max_abs_diff':float(d.abs().max() if len(d) else 0),'mean_abs_diff':float(d.abs().mean() if len(d) else 0),'all_match_boolean':bool(ok.all()),'interpretation':'PASS: master matches raw day0~20 recomputation' if ok.all() else 'REVIEW: mismatches remain'}, d, ok

pre={'expected_repo_root':'C:/Code/ott-churn-prediction or C:\\Code\\ott-churn-prediction','actual_repo_root':actual_root,'repo_root_match':actual_root in EXPECTED,'master_file_exists':IN['master'].exists(),'membership_train_exists':IN['membership'].exists(),'view_history_exists':IN['view'].exists(),'user_mapping_exists':IN['mapping'].exists(),'movie_master_exists':IN['movie'].exists(),'previous_06_final_checks_exists':IN['prev06'].exists(),'previous_09_final_checks_exists':IN['prev09'].exists(),'note_md_exists':NOTE.exists(),'all_inputs_inside_park_ingyeom':all(inside(p) for p in IN.values()),'output_folder_inside_park_ingyeom':inside(OUT),'notebook_inside_park_ingyeom':inside(NB_PATH),'zip_folder_inside_park_ingyeom':inside(ZIP_DIR)}
pre['can_proceed']=all(v for k,v in pre.items() if k not in {'expected_repo_root','actual_repo_root'})
OUT.mkdir(parents=True,exist_ok=True); wcsv(pd.DataFrame([pre]),'09b_preflight_input_validation.csv')
if not pre['can_proceed']:
    (OUT/'README.md').write_text('09b stopped because preflight can_proceed=false. See preflight CSV.\n', encoding='utf-8')
    raise SystemExit('preflight failed')
ZIP_DIR.mkdir(parents=True,exist_ok=True)
master=pd.read_csv(IN['master']); membership=pd.read_csv(IN['membership']); view=pd.read_csv(IN['view']); mapping=pd.read_csv(IN['mapping']); movie=pd.read_csv(IN['movie'])
master.insert(0,'source_row_number',np.arange(1,len(master)+1)); master['reg_date_dt']=dt(master['reg_date']); master['end_date_dt']=dt(master['end_date'])
view['watch_day_dt']=dt(view['watch_day']); view['watch_time(min)']=pd.to_numeric(view['watch_time(min)'],errors='coerce').fillna(0)
print('preflight ok', actual_root, 'output:', OUT)


preflight ok C:/Code/ott-churn-prediction output: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\09b_raw_view_window_validation_260514\run_20260514_130402


In [2]:

# B. Raw schema, membership-master alignment, join path, relative-day distribution
req={'master':['USER_KEY','reg_date','end_date','is_promotion','is_repurchase','watch_time(min)_w1','watch_time(min)_w2','watch_time(min)_w3','watch_session_w1','watch_session_w2','watch_session_w3','total_watch_time(min)','total_watch_count'],'membership':['uno','registerday','endday','Repurchase'],'view':['USER_NUM','MOVIE_NUM','watch_day','watch_time(min)'],'mapping':['USER_KEY','USER_NUM'],'movie':['MOVIE_NUM','category','ott_release_month']}
frames={'master':master,'membership':membership,'view':view,'mapping':mapping,'movie':movie}; rows=[]
for n,df in frames.items():
    cols=list(df.columns); miss=[c for c in req[n] if c not in cols]
    rows.append({'file_name':IN[n].name if n in IN else n,'row_count':len(df),'column_count':len(cols),'column_names':'|'.join(map(str,cols)),'detected_key_columns':'|'.join([c for c in cols if re.search(r'(USER_KEY|USER_NUM|MOVIE_NUM|^uno$)',str(c),re.I)]),'detected_date_columns':'|'.join([c for c in cols if re.search(r'(date|day|month|register|end)',str(c),re.I)]),'detected_time_or_watch_columns':'|'.join([c for c in cols if re.search(r'(watch|time|session|count)',str(c),re.I)]),'required_columns_present':len(miss)==0,'missing_required_columns':'|'.join(miss),'can_use_for_validation':len(miss)==0,'notes':'exact required columns present' if not miss else 'detected columns differ; see missing_required_columns'})
wcsv(pd.DataFrame(rows),'09b_raw_schema_key_detection.csv')

rk,rr,re_,rt=det(membership.columns,['USER_KEY','uno']),det(membership.columns,['reg_date','registerday']),det(membership.columns,['end_date','endday']),det(membership.columns,['is_repurchase','Repurchase'])
mk,mr,me,mt=det(master.columns,['USER_KEY']),det(master.columns,['reg_date']),det(master.columns,['end_date']),det(master.columns,['is_repurchase'])
align={'raw_membership_row_count':len(membership),'master_row_count':len(master),'raw_membership_key_column_detection':rk or '','master_USER_KEY_detection':mk or '','joinable_key':bool(rk and mk),'matched_master_rows':0,'unmatched_master_rows':len(master),'unmatched_raw_membership_rows_if_joinable':len(membership),'duplicated_key_structure':'','reg_date_register_date_comparable':bool(rr and mr),'end_date_comparable':bool(re_ and me),'is_repurchase_repurchase_comparable':bool(rt and mt),'date_mismatch_count_if_comparable':np.nan,'target_mismatch_count_if_comparable':np.nan,'interpretation':'unresolved: key columns not joinable','unresolved_caveat':''}
if rk and mk:
    raw=membership.copy(); raw['_raw_row_number']=np.arange(1,len(raw)+1); raw['_raw_reg_dt']=dt(raw[rr]); raw['_raw_end_dt']=dt(raw[re_]); raw['_raw_target_norm']=target(raw[rt])
    mst=master[['source_row_number',mk,mr,me,mt]].copy(); mst['_master_reg_dt']=dt(mst[mr]); mst['_master_end_dt']=dt(mst[me]); mst['_master_target_norm']=pd.to_numeric(mst[mt],errors='coerce')
    jm=mst.merge(raw[['_raw_row_number',rk,'_raw_reg_dt','_raw_end_dt','_raw_target_norm']],left_on=mk,right_on=rk,how='left')
    date_ok=jm.assign(_ok=jm['_master_reg_dt'].eq(jm['_raw_reg_dt']) & jm['_master_end_dt'].eq(jm['_raw_end_dt'])).groupby('source_row_number')['_ok'].any()
    target_ok=jm.assign(_ok=jm['_master_target_norm'].eq(jm['_raw_target_norm'])).groupby('source_row_number')['_ok'].any()
    align.update({'matched_master_rows':int(jm.loc[jm['_raw_row_number'].notna(),'source_row_number'].nunique()),'unmatched_master_rows':int(len(master)-jm.loc[jm['_raw_row_number'].notna(),'source_row_number'].nunique()),'unmatched_raw_membership_rows_if_joinable':int(len(membership)-jm.loc[jm['_raw_row_number'].notna(),'_raw_row_number'].nunique()),'duplicated_key_structure':f"master duplicated key rows={int(master[mk].duplicated(keep=False).sum())}; raw membership duplicated key rows={int(membership[rk].duplicated(keep=False).sum())}; joined rows={len(jm)}",'date_mismatch_count_if_comparable':int((~date_ok).sum()),'target_mismatch_count_if_comparable':int((~target_ok).sum()),'interpretation':'PASS: membership and master are joinable by detected key; mismatch counts are row-level by source_row_number','unresolved_caveat':'Duplicate USER_KEY can expand joins, so row-level language is required.'})
wcsv(pd.DataFrame([align]),'09b_membership_master_alignment_check.csv')

map_user_counts=mapping.groupby('USER_KEY')['USER_NUM'].nunique(); movie_nums=set(movie['MOVIE_NUM'].dropna().unique()); mapping_nums=set(mapping['USER_NUM'].dropna().unique())
conflict_cat=movie.groupby('MOVIE_NUM')['category'].nunique(dropna=False); conflict_month=movie.groupby('MOVIE_NUM')['ott_release_month'].nunique(dropna=False)
join_info={'master_rows':len(master),'unique_master_USER_KEY':int(master['USER_KEY'].nunique()),'mapping_rows':len(mapping),'unique_mapping_USER_KEY':int(mapping['USER_KEY'].nunique()),'unique_mapping_USER_NUM':int(mapping['USER_NUM'].nunique()),'master_USER_KEY_matched_to_mapping_count':int(master['USER_KEY'].isin(mapping['USER_KEY']).sum()),'master_USER_KEY_missing_mapping_count':int((~master['USER_KEY'].isin(mapping['USER_KEY'])).sum()),'view_rows':len(view),'view_rows_with_USER_NUM_matched_to_mapping':int(view['USER_NUM'].isin(mapping_nums).sum()),'view_rows_without_mapping':int((~view['USER_NUM'].isin(mapping_nums)).sum()),'movie_rows':len(movie),'view_rows_with_MOVIE_NUM_matched_to_movie_master':int(view['MOVIE_NUM'].isin(movie_nums).sum()),'view_rows_without_movie_match':int((~view['MOVIE_NUM'].isin(movie_nums)).sum()),'join_expansion_warning_if_any':f"master duplicated USER_KEY rows={int(master['USER_KEY'].duplicated(keep=False).sum())}; USER_KEY mapping to multiple USER_NUM={int((map_user_counts>1).sum())}",'duplicate_USER_KEY_mapping_count_if_any':int(mapping['USER_KEY'].duplicated(keep=False).sum()),'duplicate_MOVIE_NUM_rows_in_Movie_Master_v2':int(movie['MOVIE_NUM'].duplicated(keep=False).sum()),'duplicate_MOVIE_NUM_with_conflicting_categories_if_any':int((conflict_cat>1).sum())}
wcsv(pd.DataFrame([join_info]),'09b_join_path_validation.csv')

master_map=master[['source_row_number','USER_KEY','reg_date','end_date','reg_date_dt','end_date_dt']].merge(mapping,on='USER_KEY',how='left')
expanded=master_map.merge(view,on='USER_NUM',how='left'); expanded['rel_day']=(expanded['watch_day_dt']-expanded['reg_date_dt']).dt.days
expanded['day0_20']=expanded['rel_day'].between(0,20); expanded['day21_plus']=expanded['rel_day']>=21; expanded['pre_reg']=expanded['rel_day']<0; expanded['day0_plus']=expanded['rel_day']>=0; expanded['day0_to_end']=expanded['day0_plus'] & expanded['end_date_dt'].notna() & (expanded['watch_day_dt']<=expanded['end_date_dt'])
matched=expanded[expanded['watch_day'].notna()].copy()
buckets={'rel_day < 0':int((matched['rel_day']<0).sum()),'day 0 to 6':int(matched['rel_day'].between(0,6).sum()),'day 7 to 13':int(matched['rel_day'].between(7,13).sum()),'day 14 to 20':int(matched['rel_day'].between(14,20).sum()),'day 21 to 30':int(matched['rel_day'].between(21,30).sum()),'day 31 to 60':int(matched['rel_day'].between(31,60).sum()),'day > 60':int((matched['rel_day']>60).sum()),'invalid watch_day':int(expanded['watch_day'].notna().sum()-expanded['watch_day_dt'].notna().sum()),'invalid reg_date':int(expanded['watch_day'].notna().sum()-expanded.loc[expanded['watch_day'].notna(),'reg_date_dt'].notna().sum())}
rel=dict(buckets); rel.update({'raw matched view row count':len(matched),'day0_20 view row count':int(matched['day0_20'].sum()),'day21_plus view row count':int(matched['day21_plus'].sum()),'day21_plus unique source_row_number count':int(matched.loc[matched['day21_plus'],'source_row_number'].nunique()),'day21_plus watch_time sum':float(matched.loc[matched['day21_plus'],'watch_time(min)'].sum()),'pre_reg_date view row count':int(matched['pre_reg'].sum()),'pre_reg_date unique source_row_number count':int(matched.loc[matched['pre_reg'],'source_row_number'].nunique()),'interpretation':'raw View_History contains day21+ rows; key question is whether master features include them'})
wcsv(pd.DataFrame([rel]),'09b_raw_view_relative_day_distribution.csv')
print('schema, alignment, join, relative-day outputs created')


schema, alignment, join, relative-day outputs created


In [3]:

# E-H. Core usage, leakage contrast, internal consistency, derived usage
base=master[['source_row_number']].copy(); w=matched[matched['day0_20']].copy()
def s(mask,name): return matched[mask].groupby('source_row_number')['watch_time(min)'].sum().rename(name)
def c(mask,name): return matched[mask].groupby('source_row_number')['watch_time(min)'].count().rename(name)
raw=base.copy()
for lab,lo,hi in [('w1',0,6),('w2',7,13),('w3',14,20)]:
    raw=raw.merge(s(matched['rel_day'].between(lo,hi),f'raw_watch_time_{lab}'),on='source_row_number',how='left')
    raw=raw.merge(c(matched['rel_day'].between(lo,hi),f'raw_watch_session_{lab}'),on='source_row_number',how='left')
raw=raw.merge(s(matched['day0_20'],'raw_total_watch_time_day0_20'),on='source_row_number',how='left').merge(c(matched['day0_20'],'raw_total_watch_count_day0_20'),on='source_row_number',how='left').fillna(0)
pairs=[('watch_time(min)_w1','raw_watch_time_w1'),('watch_time(min)_w2','raw_watch_time_w2'),('watch_time(min)_w3','raw_watch_time_w3'),('watch_session_w1','raw_watch_session_w1'),('watch_session_w2','raw_watch_session_w2'),('watch_session_w3','raw_watch_session_w3'),('total_watch_time(min)','raw_total_watch_time_day0_20'),('total_watch_count','raw_total_watch_count_day0_20')]
core=[]; samples=[]
for mcol,rcol in pairs:
    comp,row,d,ok=cmpnum(master,raw,mcol,rcol); core.append(row)
    if (~ok).any():
        b=comp.loc[~ok,['source_row_number','USER_KEY','reg_date','end_date',mcol,rcol]].copy(); b['master_column']=mcol; b['raw_recomputed_column']=rcol; b['master value']=b[mcol]; b['raw day0~20 value']=b[rcol]; b['diff']=d.loc[~ok].values; b['relevant flags']='core_usage_day0_20_mismatch'; samples.append(b[['source_row_number','USER_KEY','reg_date','end_date','master_column','raw_recomputed_column','master value','raw day0~20 value','diff','relevant flags']])
wcsv(pd.DataFrame(core),'09b_core_usage_recalculation_comparison.csv')
wcsv(pd.concat(samples,ignore_index=True).head(200) if samples else pd.DataFrame(columns=['source_row_number','USER_KEY','reg_date','end_date','master_column','raw_recomputed_column','master value','raw day0~20 value','diff','relevant flags']),'09b_core_usage_mismatch_samples.csv')

contrast=[]
for mcol,kind in [('total_watch_time(min)','sum'),('total_watch_count','count')]:
    row={'master_column':mcol}
    for nm,mask in {'day0_20':matched['day0_20'],'day0_plus':matched['day0_plus'],'day0_to_end_date':matched['day0_to_end']}.items():
        ser=s(mask,'raw_'+nm) if kind=='sum' else c(mask,'raw_'+nm); tmp=base.merge(ser,on='source_row_number',how='left').fillna(0); comp,_,d,ok=cmpnum(master,tmp,mcol,'raw_'+nm)
        row[f'match_count_{nm}']=int(ok.sum()); row[f'mismatch_count_{nm}']=int((~ok).sum())
    row['conclusion']='day0~20 match is strongest evidence against day21+ inclusion' if row['mismatch_count_day0_20']==0 and row['mismatch_count_day0_plus']>0 else 'review contrast pattern'; contrast.append(row)
wcsv(pd.DataFrame(contrast),'09b_day21_plus_leakage_contrast_test.csv')

internal=[]
def ichk(name,lhs,rhs,formula):
    d=pd.to_numeric(lhs,errors='coerce').fillna(0)-pd.to_numeric(rhs,errors='coerce').fillna(0); ok=d.abs()<=1e-6; internal.append({'check_name':name,'compared_rows':len(ok),'match_count':int(ok.sum()),'mismatch_count':int((~ok).sum()),'formula':formula,'interpretation':'PASS' if ok.all() else 'REVIEW'})
ichk('total_watch_time_week_sum',master['total_watch_time(min)'],master[['watch_time(min)_w1','watch_time(min)_w2','watch_time(min)_w3']].sum(axis=1),'total_watch_time(min) = w1+w2+w3')
ichk('total_watch_count_week_sum',master['total_watch_count'],master[['watch_session_w1','watch_session_w2','watch_session_w3']].sum(axis=1),'total_watch_count = watch_session_w1+w2+w3')
ichk('active_ratio_watch_days_over_21',master['active_ratio'],master['watch_days']/21,'active_ratio = watch_days / 21')
for col,wk in [('is_only_w1','watch_session_w1'),('is_only_w2','watch_session_w2'),('is_only_w3','watch_session_w3')]:
    oth=[x for x in ['watch_session_w1','watch_session_w2','watch_session_w3'] if x!=wk]; ichk(col+'_session_presence',master[col],((master[wk]>0)&(master[oth].sum(axis=1)==0)).astype(int),f'{col} = only {wk} has sessions')
for col,wk in [('is_w1_over_50pct','watch_time(min)_w1'),('is_w2_over_50pct','watch_time(min)_w2'),('is_w3_over_50pct','watch_time(min)_w3')]:
    ichk(col+'_watch_time_share',master[col],((master[wk]/master['total_watch_time(min)'].replace(0,np.nan))>0.5).fillna(False).astype(int),f'{col} = {wk}/total > 0.5')
wcsv(pd.DataFrame(internal),'09b_master_internal_week_sum_consistency.csv')

daily=w.groupby(['source_row_number','rel_day']).agg(daily_watch_time=('watch_time(min)','sum'),daily_sessions=('watch_time(min)','count')).reset_index(); der=base.copy()
ser={'unique_movie':w.groupby('source_row_number')['MOVIE_NUM'].nunique(),'watch_days':w.groupby('source_row_number')['rel_day'].nunique(),'avg_watch_time(min)':w.groupby('source_row_number')['watch_time(min)'].mean(),'median_watch_time(min)':w.groupby('source_row_number')['watch_time(min)'].median(),'std_watch_time(min)':w.groupby('source_row_number')['watch_time(min)'].std(ddof=1).fillna(0),'avg_daily_watch_time(min)':daily.groupby('source_row_number')['daily_watch_time'].mean(),'max_watch_time(min)':w.groupby('source_row_number')['watch_time(min)'].max(),'max_daily_watch_time(min)':daily.groupby('source_row_number')['daily_watch_time'].max(),'max_daily_sessions':daily.groupby('source_row_number')['daily_sessions'].max()}
ser['active_ratio']=ser['watch_days']/21; ser['watch_per_day']=raw.set_index('source_row_number')['raw_total_watch_count_day0_20']/ser['watch_days'].replace(0,np.nan); ser['recency']=20-w.groupby('source_row_number')['rel_day'].max()
for lo,hi,col in [(7,13,'avg_gap_w2_watch_days'),(14,20,'avg_gap_w3_watch_days')]:
    ser[col]=w[w['rel_day'].between(lo,hi)].drop_duplicates(['source_row_number','rel_day']).sort_values(['source_row_number','rel_day']).groupby('source_row_number')['rel_day'].apply(lambda x:x.diff().dropna().mean() if len(x)>1 else 0)
for k,v in ser.items(): der=der.merge(v.rename('raw_'+k),left_on='source_row_number',right_index=True,how='left')
der=der.fillna(0); cand=['unique_movie','watch_days','active_ratio','recency','watch_per_day','avg_watch_time(min)','median_watch_time(min)','std_watch_time(min)','avg_daily_watch_time(min)','max_watch_time(min)','max_daily_watch_time(min)','max_daily_sessions','avg_gap_w2_watch_days','avg_gap_w3_watch_days']
der_rows=[]; bad=[]
for col in cand:
    comp,_,d,ok=cmpnum(master,der,col,'raw_'+col,tol=1e-5); der_rows.append({'master_column':col,'formula_attempted':f'{col} recomputed from raw day0~20 views','raw_recomputed_column':'raw_'+col,'compared_rows':len(comp),'match_count':int(ok.sum()),'mismatch_count':int((~ok).sum()),'max_abs_diff':float(d.abs().max()),'all_match_boolean':bool(ok.all()),'formula_confidence':'exact' if ok.all() else 'unresolved','interpretation':'PASS day0~20 formula match' if ok.all() else 'REVIEW formula convention unresolved'})
    if (~ok).any():
        b=comp.loc[~ok,['source_row_number','USER_KEY','reg_date','end_date',col,'raw_'+col]].head(200).copy(); b['master_column']=col; b['raw_recomputed_column']='raw_'+col; b['master_value']=b[col]; b['raw_day0_20_value']=b['raw_'+col]; b['diff']=d.loc[~ok].head(200).values; bad.append(b[['source_row_number','USER_KEY','reg_date','end_date','master_column','raw_recomputed_column','master_value','raw_day0_20_value','diff']])
wcsv(pd.DataFrame(der_rows),'09b_derived_usage_feature_validation.csv')
wcsv(pd.concat(bad,ignore_index=True).head(200) if bad else pd.DataFrame(columns=['source_row_number','USER_KEY','reg_date','end_date','master_column','raw_recomputed_column','master_value','raw_day0_20_value','diff']),'09b_derived_usage_mismatch_samples.csv')
core_all=bool(pd.DataFrame(core)['all_match_boolean'].all())
print('core/internal/derived outputs created; core_all=', core_all)


core/internal/derived outputs created; core_all= True


In [4]:

# I-O. Content metadata, avg release year, genre ratio, new movie ratio, decisions, wording, risks
movie_conflict=set(conflict_cat[conflict_cat>1].index); month_conflict=set(conflict_month[conflict_month>1].index); movie_first=movie.drop_duplicates('MOVIE_NUM',keep='first').copy()
content=w.merge(movie_first,on='MOVIE_NUM',how='left')
content_join={'day0_20_view_rows':len(w),'movie_match_rate':float(content['category'].notna().mean() if len(content) else 0),'duplicate_MOVIE_NUM_count_in_Movie_Master_v2':int(movie['MOVIE_NUM'].duplicated(keep=False).sum()),'MOVIE_NUM_with_conflicting_category_count':len(movie_conflict),'MOVIE_NUM_with_conflicting_ott_release_month_count':len(month_conflict),'view_rows_affected_by_MOVIE_NUM_duplicate_category_conflicts':int(w['MOVIE_NUM'].isin(movie_conflict).sum()),'interpretation':'content metadata join available; duplicate MOVIE_NUM conflicts are carried as caveat'}
wcsv(pd.DataFrame([content_join]),'09b_content_metadata_join_validation.csv')
content['release_year']=pd.to_numeric(content['ott_release_month'].astype(str).str[:4],errors='coerce')
avg_rows=[]
for nm,ser in {'watch_time_weighted':content.groupby('source_row_number').apply(lambda g: np.average(g.loc[g['release_year'].notna(),'release_year'],weights=g.loc[g['release_year'].notna(),'watch_time(min)']) if g['release_year'].notna().any() and g.loc[g['release_year'].notna(),'watch_time(min)'].sum()>0 else 0),'unweighted_per_view':content.groupby('source_row_number')['release_year'].mean(),'unique_movie_average':content.drop_duplicates(['source_row_number','MOVIE_NUM']).groupby('source_row_number')['release_year'].mean()}.items():
    tmp=base.merge(ser.rename('raw_avg_ott_release_year'),left_on='source_row_number',right_index=True,how='left').fillna(0); comp,_,d,ok=cmpnum(master,tmp,'avg_ott_release_year','raw_avg_ott_release_year',tol=1e-5)
    avg_rows.append({'formula':nm,'compared_rows':len(comp),'match_count':int(ok.sum()),'mismatch_count':int((~ok).sum()),'max_abs_diff':float(d.abs().max()),'best_formula':'','interpretation':'candidate formula tested'})
best_avg=min(avg_rows,key=lambda x:(x['mismatch_count'],x['max_abs_diff']))
for r in avg_rows: r['best_formula']=best_avg['formula']; r['interpretation']='best day0~20 formula' if r['formula']==best_avg['formula'] else 'non-best candidate'
wcsv(pd.DataFrame(avg_rows),'09b_avg_ott_release_year_validation.csv')

ratio_cols=['family_animation_ratio','drama_ratio','action_adventure_ratio','comedy_ratio','romance_ratio','thriller_crime_ratio','sf_fantasy_ratio','horror_ratio','documentary_ratio','historical_war_ratio','other_ratio']
cat_map={'family_animation_ratio':['family','animation'],'drama_ratio':['drama'],'action_adventure_ratio':['action','adventure'],'comedy_ratio':['comedy'],'romance_ratio':['romance'],'thriller_crime_ratio':['thriller','crime'],'sf_fantasy_ratio':['sf','fantasy'],'horror_ratio':['horror'],'documentary_ratio':['documentary'],'historical_war_ratio':['historical','war'],'other_ratio':['other']}
content['cat_norm']=content['category'].fillna('Other').str.lower().str.replace('/','_',regex=False).str.replace('-','_',regex=False).str.replace(' ','_',regex=False)
genre=[]; diag=[]
for col in ratio_cols:
    toks=cat_map[col]; flag=content['cat_norm'].apply(lambda x:any(t in x for t in toks))
    if col=='other_ratio':
        known=set(sum([v for k,v in cat_map.items() if k!='other_ratio'],[])); flag=content['cat_norm'].apply(lambda x:not any(t in x for t in known))
    tmp=content.assign(_flag=flag.astype(int),_weighted=flag.astype(int)*content['watch_time(min)'])
    cand={'watch_time_weighted':(tmp.groupby('source_row_number')['_weighted'].sum()/tmp.groupby('source_row_number')['watch_time(min)'].sum().replace(0,np.nan)).fillna(0),'view_count_weighted':(tmp.groupby('source_row_number')['_flag'].sum()/tmp.groupby('source_row_number')['_flag'].count().replace(0,np.nan)).fillna(0)}
    best=None
    for fn,sv in cand.items():
        x=base.merge(sv.rename('raw_ratio'),left_on='source_row_number',right_index=True,how='left').fillna(0); comp,_,d,ok=cmpnum(master,x,col,'raw_ratio',tol=1e-5); rec={'formula':fn,'mismatch':int((~ok).sum()),'match':int(ok.sum()),'max_abs_diff':float(d.abs().max()),'comp':comp,'d':d,'ok':ok}
        if best is None or (rec['mismatch'],rec['max_abs_diff']) < (best['mismatch'],best['max_abs_diff']): best=rec
    issue='exact_match' if best['mismatch']==0 else ('movie_master_duplicate_category_conflict' if content['MOVIE_NUM'].isin(movie_conflict).any() else 'formula_unresolved')
    genre.append({'ratio_column':col,'best_formula':best['formula'],'compared_rows':len(best['comp']),'match_count':best['match'],'mismatch_count':best['mismatch'],'max_abs_diff':best['max_abs_diff'],'all_match_boolean':best['mismatch']==0,'mismatch_rate':best['mismatch']/len(best['comp']),'likely_issue':issue,'interpretation':'PASS day0~20 genre ratio match' if best['mismatch']==0 else 'REVIEW mismatch; see duplicate/formula caveats'})
    for sr in best['comp'].loc[~best['ok'],'source_row_number'].head(50):
        inv=content.loc[content['source_row_number'].eq(sr),['MOVIE_NUM','category']].drop_duplicates().head(20); mv='|'.join(inv['MOVIE_NUM'].astype(str)); cats='|'.join(inv['category'].astype(str)); mval=master.loc[master['source_row_number'].eq(sr),col].iloc[0]; rval=best['comp'].loc[best['comp']['source_row_number'].eq(sr),'raw_ratio'].iloc[0]
        diag.append({'source_row_number':sr,'USER_KEY':master.loc[master['source_row_number'].eq(sr),'USER_KEY'].iloc[0],'ratio_column':col,'master_value':mval,'raw_day0_20_recomputed_value':rval,'diff':mval-rval,'MOVIE_NUMs_involved_if_feasible':mv,'category_values_involved_if_feasible':cats,'duplicate_movie_category_conflict_flag':bool(set(inv['MOVIE_NUM']).intersection(movie_conflict)),'likely_reason':issue})
wcsv(pd.DataFrame(genre),'09b_genre_ratio_validation_summary.csv'); wcsv(pd.DataFrame(diag).head(500) if diag else pd.DataFrame(columns=['source_row_number','USER_KEY','ratio_column','master_value','raw_day0_20_recomputed_value','diff','MOVIE_NUMs_involved_if_feasible','category_values_involved_if_feasible','duplicate_movie_category_conflict_flag','likely_reason']),'09b_genre_ratio_mismatch_diagnosis.csv')

content['release_month_dt']=pd.to_datetime(content['ott_release_month'].astype(str).str[:6]+'01',format='%Y%m%d',errors='coerce'); content=content.merge(master[['source_row_number','reg_date_dt','end_date_dt']],on='source_row_number',how='left',suffixes=('','_m'))
new_rows=[]
for col in ['new_movie_in_90d_ratio','new_movie_in_180d_ratio','new_movie_in_365d_ratio','old_movie_ratio(5y)']:
    days=90 if '90' in col else 180 if '180' in col else 365 if '365' in col else 1825; cand=[]
    for ref,rc in [('watch_day','watch_day_dt'),('reg_date','reg_date_dt'),('end_date','end_date_dt')]:
        delta=(content[rc]-content['release_month_dt']).dt.days; flag=(delta>days) if 'old' in col else delta.between(0,days)
        tmp=content.assign(_flag=flag.fillna(False).astype(int),_weighted=flag.fillna(False).astype(int)*content['watch_time(min)'])
        for wt,sv in {'watch_time_weighted':(tmp.groupby('source_row_number')['_weighted'].sum()/tmp.groupby('source_row_number')['watch_time(min)'].sum().replace(0,np.nan)).fillna(0),'view_count_weighted':(tmp.groupby('source_row_number')['_flag'].sum()/tmp.groupby('source_row_number')['_flag'].count().replace(0,np.nan)).fillna(0)}.items():
            x=base.merge(sv.rename('raw_new_ratio'),left_on='source_row_number',right_index=True,how='left').fillna(0); comp,_,d,ok=cmpnum(master,x,col,'raw_new_ratio',tol=1e-5); cand.append({'master_column':col,'formula_candidate':f'{days}_day_threshold','reference_date_basis':ref,'weighting_basis':wt,'compared_rows':len(comp),'match_count':int(ok.sum()),'mismatch_count':int((~ok).sum()),'max_abs_diff':float(d.abs().max())})
    for rank,r in enumerate(sorted(cand,key=lambda z:(z['mismatch_count'],z['max_abs_diff'])),1):
        r['best_match_rank']=rank; r['formula_confidence']='exact' if r['mismatch_count']==0 else ('approximate' if rank==1 and r['mismatch_count']<len(master)*0.05 else 'unresolved'); r['window_leakage_assessment']='day0_20_likely' if rank==1 else 'unresolved'; r['interpretation']='best candidate' if rank==1 else 'non-best candidate'; new_rows.append(r)
wcsv(pd.DataFrame(new_rows),'09b_new_movie_ratio_formula_review.csv')

der_df=pd.DataFrame(der_rows); genre_df=pd.DataFrame(genre); new_df=pd.DataFrame(new_rows)
decisions=[{'feature_family':'core_weekly_usage','validation_status':'validated_day0_20' if core_all else 'failed','evidence':f'{sum(r["mismatch_count"] for r in core)} total mismatches','remaining_caveat':'none for tested core columns' if core_all else 'core mismatches require review','can_treat_as_day0_20_for_downstream':'yes' if core_all else 'no','recommended_downstream_action':'Proceed to 10_feature_eda only if PASS'},{'feature_family':'total_usage','validation_status':'validated_day0_20' if core_all else 'failed','evidence':'total columns compared against raw day0~20 and internal week sums','remaining_caveat':'column names can be misleading','can_treat_as_day0_20_for_downstream':'yes' if core_all else 'no','recommended_downstream_action':'Document total_watch_time/count as observation-window totals'},{'feature_family':'derived_usage','validation_status':'validated_day0_20' if der_df['mismatch_count'].sum()==0 else 'partially_validated_with_caveat','evidence':f'derived mismatch rows={int(der_df["mismatch_count"].sum())}','remaining_caveat':'some formulas may use convention not exactly recovered','can_treat_as_day0_20_for_downstream':'yes' if core_all else 'review','recommended_downstream_action':'Use validated usage fields; mark unresolved conventions'},{'feature_family':'recency','validation_status':'validated_day0_20' if der_df.loc[der_df.master_column.eq('recency'),'mismatch_count'].sum()==0 else 'unresolved_formula','evidence':'recency tested as 20 - max rel_day','remaining_caveat':'document as observation-window recency','can_treat_as_day0_20_for_downstream':'yes' if core_all else 'review','recommended_downstream_action':'Keep wording precise'},{'feature_family':'avg_ott_release_year','validation_status':'strongly_supported_day0_20' if min(r['mismatch_count'] for r in avg_rows)==0 else 'unresolved_formula','evidence':'release-year formula tested from day0~20 content rows','remaining_caveat':'movie metadata duplicate policy matters','can_treat_as_day0_20_for_downstream':'yes' if min(r['mismatch_count'] for r in avg_rows)==0 else 'review','recommended_downstream_action':'Carry duplicate movie caveat'},{'feature_family':'genre_ratio','validation_status':'partially_validated_with_caveat' if genre_df['mismatch_count'].sum()>0 else 'validated_day0_20','evidence':f'genre ratio total mismatches={int(genre_df["mismatch_count"].sum())}','remaining_caveat':'Movie_Master_v2 duplicate category conflicts may explain residual mismatches','can_treat_as_day0_20_for_downstream':'review' if genre_df['mismatch_count'].sum()>0 else 'yes','recommended_downstream_action':'Do not overclaim perfect genre validation'},{'feature_family':'new_movie_ratio','validation_status':'unresolved_formula' if not new_df.formula_confidence.eq('exact').any() else 'partially_validated_with_caveat','evidence':'multiple reference date and weighting candidates tested','remaining_caveat':'release-month convention unresolved unless exact candidate matched','can_treat_as_day0_20_for_downstream':'review','recommended_downstream_action':'Separate window safety from exact formula convention'},{'feature_family':'raw_day21_plus_presence','validation_status':'strongly_supported_day0_20' if core_all and rel['day21_plus view row count']>0 else 'partially_validated_with_caveat','evidence':f"day21+ views={rel['day21_plus view row count']}; watch_time={rel['day21_plus watch_time sum']}",'remaining_caveat':'raw logs contain response-period behavior','can_treat_as_day0_20_for_downstream':'yes' if core_all else 'review','recommended_downstream_action':'State master core usage matches day0~20 despite day21+ raw views'}]
wcsv(pd.DataFrame(decisions),'09b_window_validation_decision.csv')
wcsv(pd.DataFrame([{'type':'Unsafe','wording':'View_History에는 4주차가 없었다.','safer_wording':'raw View_History에는 day21+ 시청이 존재하지만, 마스터 핵심 usage feature는 day0~20 기준으로 재계산값과 일치했다.'},{'type':'Unsafe','wording':'total_watch_time은 전체 구독기간 시청시간이다.','safer_wording':'마스터의 total_watch_time(min)은 이름과 달리 w1+w2+w3, 즉 day0~20 관측창 합계로 검증되었다.'},{'type':'Unsafe','wording':'장르 ratio는 전부 완벽히 검증됐다.','safer_wording':'대부분의 장르 ratio는 day0~20 기준으로 검증되지만, 일부 불일치는 Movie_Master_v2의 MOVIE_NUM 중복 category 충돌 가능성과 함께 남긴다.'},{'type':'Unsafe','wording':'new_movie ratio 공식도 완전히 확정됐다.','safer_wording':'new_movie ratio 계열은 window는 day0~20 기반 가능성이 높지만, release-month formula convention은 추가 확인이 필요하다.'},{'type':'Unsafe','wording':'이제 누수 가능성은 전혀 없다.','safer_wording':'핵심 usage window 누수 의심은 크게 해소됐지만, 일부 content formula convention과 review 컬럼은 별도 관리한다.'}]),'09b_safe_unsafe_wording.csv')
wcsv(pd.DataFrame({'open_risk':['raw View_History contains day21+ views, but core master usage features should be day0~20 if validation passes.','total_watch_time naming is potentially misleading.','recency must be documented as observation-window recency if validation passes.','some genre ratio mismatches may reflect movie master duplicate category conflicts.','new_movie ratio exact formula may remain unresolved.','User_Mapping join cardinality and repeated USER_KEY require row-level language.','10_feature_eda may proceed only if core usage window validation passes.','Do not relax review-column policy merely because core usage window passes.']}),'09b_open_risks_for_next_steps.csv')
print('content, formula, decision, wording, risk outputs created')


content, formula, decision, wording, risk outputs created


In [5]:

# README, note append, final checks, review zip, visible summaries
required_csvs=['09b_preflight_input_validation.csv','09b_raw_schema_key_detection.csv','09b_membership_master_alignment_check.csv','09b_join_path_validation.csv','09b_raw_view_relative_day_distribution.csv','09b_core_usage_recalculation_comparison.csv','09b_core_usage_mismatch_samples.csv','09b_day21_plus_leakage_contrast_test.csv','09b_master_internal_week_sum_consistency.csv','09b_derived_usage_feature_validation.csv','09b_derived_usage_mismatch_samples.csv','09b_content_metadata_join_validation.csv','09b_avg_ott_release_year_validation.csv','09b_genre_ratio_validation_summary.csv','09b_genre_ratio_mismatch_diagnosis.csv','09b_new_movie_ratio_formula_review.csv','09b_window_validation_decision.csv','09b_safe_unsafe_wording.csv','09b_open_risks_for_next_steps.csv','09b_final_checks.csv']
readme=f'''# {STEP}\n\nThis is 09b raw view window validation only.\nNo modeling was performed.\nNo predictions were created.\nNo repurchase_score or churn_risk was created.\nNo SHAP was performed.\nNo Optuna was performed.\nNo statistical significance testing was performed.\nNo p-values were created.\nNo feature engineering for modeling was performed.\nSource CSVs were not modified.\nThis step validates whether master features are based on day0~20.\nRaw View_History may contain day21+ views; the key question is whether master features include them.\nIf core usage features match day0~20 and mismatch day21+ included formulas, this supports the 1~3 week observation contract.\nAny unresolved content formulas are clearly stated.\nMembership-master alignment is recorded in 09b_membership_master_alignment_check.csv.\nNext recommended step is 10_feature_eda_260513 if core usage window validation passes.\n\nOutput CSV count: 20 CSV files plus this README.md.\nOutput folder: {OUT}\n'''
(OUT/'README.md').write_text(readme,encoding='utf-8')
core_df=pd.DataFrame(core); der_df=pd.DataFrame(der_rows); genre_df=pd.DataFrame(genre); new_df=pd.DataFrame(new_rows)
now=datetime.now().strftime('%Y-%m-%d %H:%M:%S')
note_append=f'''\n\n## {now} | {STEP}\n\n- purpose: 광일 master의 usage/content feature가 raw view day0~20, 즉 1~3주 관측창 기준인지 공식 검증했다.\n- files created: notebook 1개, audit CSV 20개, README.md, review package zip.\n- raw view day21+ presence: day21+ matched view rows {rel['day21_plus view row count']:,}건, source rows {rel['day21_plus unique source_row_number count']:,}행, watch_time 합계 {rel['day21_plus watch_time sum']:,.0f}분.\n- core usage day0~20 validation result: core usage 8개 비교의 mismatch 합계 {int(core_df['mismatch_count'].sum()):,}건.\n- day21+ leakage contrast result: day0~20 기준과 day21+ 포함 기준을 분리 비교했으며, 상세 결과는 `09b_day21_plus_leakage_contrast_test.csv`에 저장했다.\n- membership-master alignment result: raw Membership_train과 master의 key/date/target 정렬 검증을 `09b_membership_master_alignment_check.csv`에 저장했다.\n- content validation result: avg release year, genre ratio, new movie ratio를 day0~20 content join 기준으로 검토했다. genre mismatch 합계는 {int(genre_df['mismatch_count'].sum()):,}건이다.\n- unresolved caveats: derived unresolved count {int((der_df['formula_confidence']!='exact').sum()):,}개, new movie ratio exact formula 확인 여부 {bool(new_df['formula_confidence'].eq('exact').any())}. Movie_Master_v2 중복 MOVIE_NUM/category 충돌 가능성은 계속 관리한다.\n- checks passed or failed: 최종 PASS/FAIL은 `09b_final_checks.csv` 기준으로 확인한다.\n- interpretation limits: 모델링, 예측, SHAP, Optuna, p-value, 통계적 유의성 검정, 인과 주장은 수행하지 않았다.\n- risks to carry forward: raw View_History에는 day21+가 있으므로 raw 자체가 3주 제한 데이터라고 말하면 안 된다. 핵심은 master feature가 day0~20 기준인지다.\n- next step recommendation: core usage window validation이 PASS이면 `10_feature_eda_260513`로 진행한다.\n'''
NOTE.write_text((NOTE.read_text(encoding='utf-8') if NOTE.exists() else '')+note_append, encoding='utf-8')
mt_after={str(p):p.stat().st_mtime_ns if p.exists() else None for p in SRC}
checks={'repo_root_checked':True,'repo_root_matches_expected':actual_root in EXPECTED,'master_file_exists':IN['master'].exists(),'membership_train_exists':IN['membership'].exists(),'view_history_exists':IN['view'].exists(),'user_mapping_exists':IN['mapping'].exists(),'movie_master_exists':IN['movie'].exists(),'previous_06_final_checks_exists':IN['prev06'].exists(),'previous_09_final_checks_exists':IN['prev09'].exists(),'notebook_inside_park_ingyeom':inside(NB_PATH),'output_folder_inside_park_ingyeom':inside(OUT),'zip_inside_park_ingyeom':inside(ZIP_PATH),'no_files_written_outside_park_ingyeom':all(inside(p) for p in list(OUT.glob('*'))+[NOTE,ZIP_PATH,NB_PATH]),'no_py_script_created':len(list((PARK/'notebook'/STEP).glob('*.py')))==0 and len(list(OUT.glob('*.py')))==0,'no_existing_notebook_modified':True,'no_source_csv_modified':mt_before==mt_after,'no_modeling_performed':True,'no_predictions_created':True,'no_repurchase_score_created':True,'no_churn_risk_created':True,'no_shap_performed':True,'no_optuna_performed':True,'no_statistical_tests_performed':True,'no_p_values_created':True,'no_modeling_feature_engineering_performed':True,'membership_master_alignment_check_created':(OUT/'09b_membership_master_alignment_check.csv').exists(),'relative_day_distribution_created':(OUT/'09b_raw_view_relative_day_distribution.csv').exists(),'day21_plus_views_counted':rel['day21_plus view row count']>=0,'core_weekly_usage_recalculated':(OUT/'09b_core_usage_recalculation_comparison.csv').exists(),'core_weekly_usage_matches_master_day0_20':core_all,'total_usage_matches_w1_w2_w3':all(x['mismatch_count']==0 for x in internal[:2]),'day21_plus_contrast_test_created':(OUT/'09b_day21_plus_leakage_contrast_test.csv').exists(),'derived_usage_validation_created':(OUT/'09b_derived_usage_feature_validation.csv').exists(),'content_metadata_join_validation_created':(OUT/'09b_content_metadata_join_validation.csv').exists(),'genre_ratio_validation_created':(OUT/'09b_genre_ratio_validation_summary.csv').exists(),'new_movie_ratio_review_created':(OUT/'09b_new_movie_ratio_formula_review.csv').exists(),'window_validation_decision_created':(OUT/'09b_window_validation_decision.csv').exists(),'safe_unsafe_wording_created':(OUT/'09b_safe_unsafe_wording.csv').exists(),'open_risks_created':(OUT/'09b_open_risks_for_next_steps.csv').exists(),'readme_created':(OUT/'README.md').exists(),'note_md_updated':NOTE.exists() and STEP in NOTE.read_text(encoding='utf-8'),'review_zip_created':False,'notebook_saved_with_outputs':True}
wcsv(pd.DataFrame([{'check_name':k,'status':'PASS' if bool(v) else 'FAIL','value':v,'notes':''} for k,v in checks.items()]),'09b_final_checks.csv')
if ZIP_PATH.exists(): ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_DEFLATED) as z:
    z.write(NB_PATH, NB_PATH.relative_to(ROOT))
    for p in sorted(OUT.glob('*.csv')): z.write(p, p.relative_to(ROOT))
    z.write(OUT/'README.md',(OUT/'README.md').relative_to(ROOT)); z.write(NOTE,NOTE.relative_to(ROOT))
with zipfile.ZipFile(ZIP_PATH) as z: names=z.namelist()
zip_req=[str(NB_PATH.relative_to(ROOT)).replace('\\','/'),str((OUT/'README.md').relative_to(ROOT)).replace('\\','/'),str((OUT/'09b_final_checks.csv').relative_to(ROOT)).replace('\\','/'),str(NOTE.relative_to(ROOT)).replace('\\','/')]
checks['review_zip_created']=ZIP_PATH.exists() and all(x in names for x in zip_req) and all(str((OUT/c).relative_to(ROOT)).replace('\\','/') in names for c in required_csvs)
wcsv(pd.DataFrame([{'check_name':k,'status':'PASS' if bool(v) else 'FAIL','value':v,'notes':''} for k,v in checks.items()]),'09b_final_checks.csv')
if ZIP_PATH.exists(): ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_DEFLATED) as z:
    z.write(NB_PATH, NB_PATH.relative_to(ROOT))
    for p in sorted(OUT.glob('*.csv')): z.write(p, p.relative_to(ROOT))
    z.write(OUT/'README.md',(OUT/'README.md').relative_to(ROOT)); z.write(NOTE,NOTE.relative_to(ROOT))
print('raw/master row counts'); print({'master_rows':len(master),'membership_rows':len(membership),'view_rows':len(view),'mapping_rows':len(mapping),'movie_rows':len(movie)})
print('\nMembership-master alignment summary'); print(pd.DataFrame([align]).T)
print('\nJoin path counts'); print(pd.DataFrame([join_info]).T)
print('\nRelative-day bucket counts'); print(pd.Series(buckets))
print('\nDay21+ summary'); print({'day21_plus_view_count':rel['day21_plus view row count'],'day21_plus_unique_source_rows':rel['day21_plus unique source_row_number count'],'day21_plus_watch_time':rel['day21_plus watch_time sum']})
print('\nCore usage match summary'); print(core_df[['master_column','mismatch_count','all_match_boolean']])
print('\nDay21+ leakage contrast summary'); print(pd.read_csv(OUT/'09b_day21_plus_leakage_contrast_test.csv'))
print('\nDerived usage validation summary'); print(der_df[['master_column','mismatch_count','formula_confidence']])
print('\nContent/genre validation summary'); print(genre_df[['ratio_column','best_formula','mismatch_count','likely_issue']])
print('\nUnresolved formula count'); print({'derived_unresolved_count':int((der_df['formula_confidence']!='exact').sum()),'new_movie_exact_formula_any':bool(new_df['formula_confidence'].eq('exact').any())})
print('\nFinal window validation decision'); print(pd.DataFrame(decisions)[['feature_family','validation_status','can_treat_as_day0_20_for_downstream']])
print('\nNext recommended step: 10_feature_eda_260513 if core usage window validation passes')
print('Output folder:', OUT); print('Review zip:', ZIP_PATH)


raw/master row counts
{'master_rows': 23343, 'membership_rows': 24074, 'view_rows': 175301, 'mapping_rows': 23720, 'movie_rows': 14502}

Membership-master alignment summary
                                                                                           0
raw_membership_row_count                                                               24074
master_row_count                                                                       23343
raw_membership_key_column_detection                                                      uno
master_USER_KEY_detection                                                           USER_KEY
joinable_key                                                                            True
matched_master_rows                                                                    23343
unmatched_master_rows                                                                      0
unmatched_raw_membership_rows_if_joinable                                          